In [ ]:
# 第 1 格：Notebook 内核自检。
# 这一格不导入 faas-sim，不运行 subprocess，只验证 Jupyter 内核是否真的在执行代码。
# 如果这一格都没有输出，问题就不在 faas-sim，也不在 watchdog 示例，而在 Jupyter 内核或前端显示。

print("【第 1 格】Notebook 内核已开始执行。", flush=True)

import sys
import os
from pathlib import Path

print("Python 可执行文件：", sys.executable, flush=True)
print("当前工作目录：", Path.cwd().resolve(), flush=True)
print("操作系统工作目录：", os.getcwd(), flush=True)
print("【第 1 格】执行完成。", flush=True)

# `examples/watchdogs` Watchdog 示例首格代码自检版

这个版本把**第一格直接设置为代码单元**，并且第一格只做最小自检：

```python
print("Notebook 内核已开始执行")
```

因此：

- 如果第 1 格能输出，说明 Jupyter 内核和输出区正常；
- 如果第 1 格没有任何输出，说明问题不在 faas-sim、watchdogs 或 subprocess，而是 Jupyter 内核没有真正执行，或者前端没有显示输出。

通过第 1 格自检后，再继续运行后面的项目路径检测和样例脚本运行单元。

## 2. 定位 faas-sim 项目根目录

In [ ]:
# 第 2 格：定位项目根目录。
# 判断标准：目录中同时存在 sim/ 和 examples/。

from pathlib import Path
import sys
import os
import subprocess

print("【第 2 格】开始定位 faas-sim 项目根目录。", flush=True)

current_dir = Path.cwd().resolve()

candidate_roots = [
    current_dir,
    current_dir.parent,
    current_dir.parent.parent,
    current_dir.parent.parent.parent,
]

PROJECT_ROOT = None
for root in candidate_roots:
    print("检查候选目录：", root, flush=True)
    if (root / "sim").exists() and (root / "examples").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "没有找到 faas-sim 项目根目录。请把 Notebook 放在 faas-sim 项目根目录，"
        "或者放在 examples/watchdogs/ 目录附近运行。"
    )

script_path = PROJECT_ROOT / "examples" / "watchdogs" / "main.py"

print("当前 Notebook 工作目录：", current_dir, flush=True)
print("faas-sim 项目根目录：", PROJECT_ROOT, flush=True)
print("watchdogs 样例脚本：", script_path, flush=True)

if not script_path.exists():
    raise FileNotFoundError(f"找不到样例脚本：{script_path}")

print("【第 2 格】项目根目录定位完成。", flush=True)

## 3. 运行原始 `examples/watchdogs/main.py`

这一格使用当前 Jupyter 内核对应的 Python 执行原始 `.py` 文件，等价于：

```bash
python -u examples/watchdogs/main.py
```

为了避免 Windows + Jupyter 下逐行管道读取卡住，这里使用 `subprocess.run`，脚本结束后一次性打印完整日志，并设置 60 秒超时。

In [ ]:
# 第 3 格：运行原始 watchdogs 样例脚本。

print("【第 3 格】准备运行 watchdogs 样例。", flush=True)
print("当前 Jupyter 内核 Python：", sys.executable, flush=True)

env = os.environ.copy()

# 确保子进程优先从项目根目录导入本地包。
existing_pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = (
    str(PROJECT_ROOT)
    if not existing_pythonpath
    else str(PROJECT_ROOT) + os.pathsep + existing_pythonpath
)

# 减少 Windows/Jupyter 输出编码差异。
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONIOENCODING"] = "utf-8"

cmd = [
    sys.executable,
    "-u",
    str(script_path),
]

print("执行命令：", " ".join(cmd), flush=True)
print("工作目录：", PROJECT_ROOT, flush=True)
print("开始运行子进程，最多等待 60 秒。", flush=True)

try:
    result = subprocess.run(
        cmd,
        cwd=str(PROJECT_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=60,
        check=False,
    )

    print("\n========== 子进程完整输出开始 ==========", flush=True)
    print(result.stdout, flush=True)
    print("========== 子进程完整输出结束 ==========", flush=True)

    print("子进程退出码：", result.returncode, flush=True)

    if result.returncode != 0:
        raise RuntimeError(f"watchdogs 样例运行失败，退出码：{result.returncode}")

    print("【第 3 格】watchdogs 样例运行完成。", flush=True)

except subprocess.TimeoutExpired as e:
    print("\n========== 子进程超时，已捕获输出开始 ==========", flush=True)

    if e.stdout:
        if isinstance(e.stdout, bytes):
            print(e.stdout.decode("utf-8", errors="replace"), flush=True)
        else:
            print(e.stdout, flush=True)

    if e.stderr:
        if isinstance(e.stderr, bytes):
            print(e.stderr.decode("utf-8", errors="replace"), flush=True)
        else:
            print(e.stderr, flush=True)

    print("========== 子进程超时，已捕获输出结束 ==========", flush=True)
    raise RuntimeError(
        "watchdogs 样例在 Notebook 中运行超过 60 秒。"
        "如果第 1 格和第 2 格正常，重点检查当前 Jupyter 内核 Python 是否与 PowerShell 运行样例的 Python 一致。"
    )

## 4. 判断标准

如果第 1 格没有任何输出：

```text
【第 1 格】Notebook 内核已开始执行。
```

说明 Notebook 内核没有正常执行，或者输出区没有刷新。这时不需要继续排查 faas-sim。

如果第 1 格有输出，但第 2 格没有输出，说明当前 Notebook 前端或内核状态异常，建议重启内核后再运行。

如果第 1、2 格都正常，第 3 格异常，再看第 3 格打印的 Python 路径、工作目录和子进程日志。